# Agents at Scale: Multi-Agent Architecture with A2A Protocol on Agent Runtime and ADK Integration

## Overview

As AI agents take on more responsibilities, a single agent doing everything becomes hard to maintain, scale, and evolve. Different capabilities often need different deployment strategies, update cycles, or even different teams owning them.

* The [A2A (Agent2Agent) Protocol](https://a2a-protocol.org/latest/) solves the communication side — standardizing how agents discover each other's capabilities and collaborate across frameworks and organizations.
* [Gemini Enterprise Agent Platform Runtime](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) solves the deployment side — a fully managed, serverless platform that hosts your agents with built-in A2A support, auto-scaling, secure endpoints, persistent sessions, and zero infrastructure management.

Together, they let you build specialized agents, deploy them as discoverable A2A services, and compose them into multi-agent systems.

## What You'll Build

A **Reservation Agent** that manages restaurant table bookings (create, check, and cancel) using ADK session state which is managed by Gemini Enterprise Agent Platform Sessions. You deploy this agent to Gemini Enterprise Agent Platform Runtime where it becomes discoverable via the A2A protocol's agent card. Then you upgrade the **Foodie Finds** restaurant concierge agent to consume the Reservation Agent as a remote A2A sub-agent. The result: a multi-agent system where the orchestrator routes menu queries to MCP Toolbox and reservation requests to the remote A2A agent.

## What You'll Learn

* Build an ADK agent that uses managed session service to manage reservation data without an external database.
* Expose an ADK agent as an A2A server with agent cards and skills.
* Deploy an A2A agent to Gemini Enterprise Agent Runtime.
* Consume a remote A2A agent from another ADK agent using `RemoteA2aAgent` and handle authenticated requests.
* Test multi-agent systems incrementally: local A2A, deployed A2A, partial integration, and full deployment.

## Environment Setup

We import required packages and set up our Google Cloud project and location environment variables.
In this tutorial, we will write our modular agent packages (`reservation_agent`, `restaurant_agent`, and `scripts`) into a local `./a2a_agent` directory using `%%writefile`.

In [ ]:
import asyncio
import logging
import os
import sys
import uuid
import warnings

import google.auth
import vertexai

from a2a.helpers import new_text_message
from a2a.server.context import ServerCallContext
from a2a.types import (
    AgentCard,
    AgentSkill,
    GetExtendedAgentCardRequest,
    GetTaskRequest,
    Message,
    Part,
    Role,
    SendMessageRequest,
    TaskState,
)
from google.adk.a2a.utils.agent_to_a2a import (
    A2aAgentExecutor,
    AgentCardBuilder,
    to_a2a,
)
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.auth import default
from google.genai import Client, types
from vertexai.agent_engines.templates.a2a import A2aAgent, create_agent_card

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

In [ ]:
# Set Google Cloud Project and Location
_, PROJECT_ID = google.auth.default()
LOCATION = "global"
REGION = "us-central1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["REGION"] = REGION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Agent Platform API

In [ ]:
# Create directory structure for our modular agent files
!mkdir -p ./a2a_agent/reservation_agent ./a2a_agent/restaurant_agent

## Concept: Agent2Agent (A2A) Protocol and Gemini Enterprise Agent Runtime

### The Agent2Agent (A2A) Protocol
The [Agent2Agent (A2A) protocol](https://a2a-protocol.org) is an open standard designed to enable seamless communication and collaboration between AI agents. Where MCP (Model Context Protocol) connects agents to *tools and data*, A2A connects agents to *other agents* — enabling them to discover each other's capabilities, delegate tasks, and collaborate across frameworks and organizations.

The key difference between wrapping an agent as a tool (via MCP) vs exposing it via A2A: tools are stateless and perform single functions, while A2A agents can reason, maintain state, and handle multi-turn interactions like negotiation or clarification. An agent exposed via A2A retains its full capabilities rather than being reduced to a function call.

A2A defines three core concepts:
1. **Agent Card** — a JSON document describing what an agent does, its skills, and its endpoint. Other agents fetch this card to discover capabilities.
2. **Message** — a user or agent request sent to an A2A endpoint, triggering a task.
3. **Task** — a unit of work with a lifecycle (submitted → working → completed/failed) and **artifacts** containing the results.

### Gemini Enterprise Agent Platform Runtime
**Agent Runtime** is a fully managed service on Google Cloud for deploying, scaling, and managing AI agents in production with Enterprise security features (e.g. VPC Service Controls, CMEK). It handles infrastructure so you can focus on agent logic.

Agent Runtime provides:
* **Managed deployment** — deploy agents built with ADK, LangGraph, or any Python framework with a single SDK call
* **A2A hosting** — deploy agents as A2A-compliant endpoints with automatic agent card serving and authenticated access
* **Persistent sessions** — `VertexAiSessionService` stores conversation history and state across requests
* **Auto-scaling** — scales from zero to handle traffic, with no infrastructure management
* **Observability** — built-in tracing, logging, and monitoring via Google Cloud's observability stack

In this tutorial, you deploy the reservation agent to Agent Runtime. The deployment process serializes (pickles) your agent code and uploads it. Agent Runtime provisions a serverless endpoint that serves the A2A protocol — other agents (or clients) interact with it via standard HTTP calls, authenticated with Google Cloud credentials.

## Building and Exposing the Reservation Agent


### Build the Reservation Agent

This step creates a new ADK agent that handles restaurant reservations using session state (`ToolContext`). The agent supports three operations — create, check, and cancel — with the phone number as the lookup key. All reservation data lives in ADK's session state (`app:reservation:{phone_number}`).

We use `%%writefile` to save our package init and agent implementation into `./a2a_agent/reservation_agent/`.

In [ ]:
%%writefile ./a2a_agent/reservation_agent/__init__.py
# Package init for reservation_agent


In [ ]:
%%writefile ./a2a_agent/reservation_agent/agent.py
# reservation_agent/agent.py
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import ToolContext

# App-scoped state prefix ensures reservations persist across all sessions.
# See https://adk.dev/sessions/state/ for state scope details.
STATE_PREFIX = "app:reservation:"


def create_reservation(
    phone_number: str,
    name: str,
    party_size: int,
    date: str,
    time: str,
    tool_context: ToolContext,
) -> dict:
    """Create a new restaurant reservation.

    Args:
        phone_number: Customer's phone number, used as the reservation ID.
        name: Name for the reservation.
        party_size: Number of guests.
        date: Reservation date (e.g., '2025-07-15' or 'this Friday').
        time: Reservation time (e.g., '7:00 PM').

    Returns:
        Confirmation of the reservation.
    """
    reservation = {
        "name": name,
        "party_size": party_size,
        "date": date,
        "time": time,
        "status": "confirmed",
    }
    tool_context.state[f"{STATE_PREFIX}{phone_number}"] = reservation
    return {
        "status": "confirmed",
        "message": f"Reservation created for {name}, party of {party_size} on {date} at {time}. Phone: {phone_number}.",
    }


def check_reservation(phone_number: str, tool_context: ToolContext) -> dict:
    """Look up an existing reservation by phone number.

    Args:
        phone_number: The phone number used when the reservation was created.
        tool_context: ADK tool context for state access.

    Returns:
        The reservation details, or a message if not found.
    """
    reservation = tool_context.state.get(f"{STATE_PREFIX}{phone_number}")
    if reservation:
        return {"found": True, "reservation": reservation}
    return {"found": False, "message": f"No reservation found for {phone_number}."}


def cancel_reservation(phone_number: str, tool_context: ToolContext) -> dict:
    """Cancel an existing reservation by phone number.

    Args:
        phone_number: The phone number used when the reservation was created.
        tool_context: ADK tool context for state access.

    Returns:
        Confirmation of cancellation, or a message if not found.
    """
    key = f"{STATE_PREFIX}{phone_number}"
    reservation = tool_context.state.get(key)
    if not reservation:
        return {
            "success": False,
            "message": f"No reservation found for {phone_number}.",
        }
    if reservation.get("status") == "cancelled":
        return {
            "success": False,
            "message": f"Reservation for {phone_number} is already cancelled.",
        }
    reservation["status"] = "cancelled"
    tool_context.state[key] = reservation
    return {
        "success": True,
        "message": f"Reservation for {reservation['name']} ({phone_number}) has been cancelled.",
    }


root_agent = LlmAgent(
    name="reservation_agent",
    description="Handles restaurant table reservations — create, check, and cancel bookings for Foodie Finds restaurant.",
    model=Gemini(
        model="gemini-3.5-flash",
        client_kwargs={"vertexai": True, "location": "global"},
    ),
    instruction="""You are a friendly reservation assistant for "Foodie Finds" restaurant.
You help diners create, check, and cancel table reservations.

When a diner wants to make a reservation, collect these details:
- Name for the reservation
- Phone number (used as the reservation ID)
- Party size (number of guests)
- Date
- Time

When all required reservation details are provided, immediately execute the create_reservation tool without asking for extra confirmation.
When checking or cancelling, ask for the phone number if not provided.
Be concise and professional.""",
    tools=[create_reservation, check_reservation, cancel_reservation],
)


### Convert to an A2A ASGI Application (`to_a2a`) and Explore Local Testing

ADK provides a built-in utility function `to_a2a(root_agent)` (`from google.adk.a2a.utils.agent_to_a2a import to_a2a, AgentCardBuilder`) that converts any ADK agent into an A2A-compliant ASGI Starlette application without requiring a custom configuration file or custom executor boilerplate.

### Why `to_a2a()`?
1. **Zero Boilerplate**: Automatically creates an under-the-hood `A2aAgentExecutor` backed by ADK runners and in-memory services.
2. **Automatic Agent Card Generation**: Extracts the agent's name, description, and tools into a standard `.well-known/agent-card.json` card automatically.
3. **Local Serving & Debugging**: Can be served immediately via `uvicorn agent:a2a_app --port 8001` or CLI (`adk api_server --a2a`) and tested with `adk web`.

In [ ]:
# Step 2: Explore ADK's native A2A ASGI Application and auto-generated Agent Card
if "./a2a_agent" not in sys.path:
    sys.path.insert(0, "./a2a_agent")

from a2a_agent.reservation_agent.agent import root_agent

# 1. Convert the ADK agent directly to an A2A ASGI Starlette application
#    In a standalone server, you can serve this app using:
#    uvicorn reservation_agent.agent:a2a_app --host localhost --port 8001
a2a_app = to_a2a(root_agent, host="localhost", port=8001, protocol="http")

# 2. Inspect the auto-generated Agent Card using AgentCardBuilder
card_builder = AgentCardBuilder(
    agent=root_agent, rpc_url="http://localhost:8001/"
)
auto_card = await card_builder.build()

print("=== ADK Auto-Generated Agent Card ===")
print(f"Name: {auto_card.name}")
print(f"Description: {auto_card.description}")
print(f"Skills: {[skill.name for skill in auto_card.skills]}")
print(
    f"Protocol Binding: {auto_card.supported_interfaces[0].protocol_binding}"
)
print(f"RPC URL: {auto_card.supported_interfaces[0].url}")

### Prepare the A2A Agent for Cloud Deployment (`A2aAgent` + `A2aAgentExecutor`) and Test Locally

While `to_a2a()` creates an ASGI application for self-hosting (such as on Cloud Run or a local container), **Google Cloud Agent Runtime** (`client.agent_engines.create(...)`) expects the `vertexai.agent_engines.templates.a2a.A2aAgent` template class with an `HTTP+JSON` interface binding.

Instead of writing a custom executor from scratch, we use ADK's built-in **`A2aAgentExecutor`** (`from google.adk.a2a.utils.agent_to_a2a import A2aAgentExecutor`) as the executor builder for `A2aAgent`.

In this step, we wrap our reservation agent using `A2aAgent` + `A2aAgentExecutor` and test the complete A2A protocol lifecycle locally inside the notebook:
1. **Agent Card Retrieval** (`on_get_extended_agent_card`) — discovering capabilities and skills.
2. **Message Sending** (`on_message_send`) — submitting a request to create a task.
3. **Task Status & Artifacts** (`on_get_task`) — polling for task completion and extracting results.
4. **Session Continuity** — reusing `contextId` across requests so the agent remembers previous turns.

#### Define Helper Functions for Mock A2A Requests

In standard A2A deployments, the Agent Runtime server converts incoming HTTP requests into Starlette `Request` objects. To test our `A2aAgent` locally without spinning up an external web server, we define helper functions (`build_get_request`, `build_post_request`, `wait_for_task`) that simulate incoming HTTP requests and poll task status.

In [ ]:
# Helper functions to test A2A protocol methods locally using SDK protobuf types
def create_call_context() -> ServerCallContext:
    """Creates a default ServerCallContext for local method invocation."""
    return ServerCallContext()


def build_send_message_request(text: str, context_id: str = "") -> SendMessageRequest:
    """Builds a SendMessageRequest with a user text message."""
    return SendMessageRequest(
        message=Message(
            message_id=f"msg-{uuid.uuid4().hex[:8]}",
            role=Role.ROLE_USER,
            parts=[Part(text=text)],
            context_id=context_id,
        )
    )


async def wait_for_task(a2a_agent, task_id: str, context: ServerCallContext, max_retries: int = 30):
    """Poll on_get_task until the task reaches completed or failed state."""
    for _ in range(max_retries):
        result = await a2a_agent.on_get_task(
            request=GetTaskRequest(id=task_id),
            context=context,
        )
        if result and result.status.state in [
            TaskState.TASK_STATE_COMPLETED,
            TaskState.TASK_STATE_FAILED,
        ]:
            return result
        await asyncio.sleep(1)
    return result


def print_task_answer(result):
    """Extract and print the text response from task artifacts."""
    if not result:
        print("No task result returned.")
        return
    print(f"Status: {TaskState.Name(result.status.state)}")
    for artifact in result.artifacts:
        for part in artifact.parts:
            if part.text:
                print(f"Answer: {part.text}")

#### Initialize the Local A2A Agent

We instantiate `A2aAgent` passing our `agent_card` and ADK's built-in `A2aAgentExecutor`, then call `set_up()` to initialize the ADK runner and in-memory session service.

In [ ]:
# Ensure ./a2a_agent is in sys.path to import our modules
if "./a2a_agent" not in sys.path:
    sys.path.insert(0, "./a2a_agent")

from a2a_agent.reservation_agent.agent import root_agent

# 1. Prepare a function builder for the ADK Runner so the agent template remains picklable for deployment
def get_reservation_runner():
    from a2a_agent.reservation_agent.agent import root_agent
    return Runner(
        app_name=root_agent.name,
        agent=root_agent,
        session_service=InMemorySessionService(),
    )

# 2. Build the AgentCard for Google Cloud Agent Runtime (requires HTTP+JSON binding)
skills = [
    AgentSkill(
        id=tool.__name__,
        name=tool.__name__,
        description=tool.__doc__ or "",
        tags=["reservations", "restaurant"],
        input_modes=["text/plain"],
        output_modes=["text/plain"],
    )
    for tool in root_agent.tools
]
agent_card = create_agent_card(
    agent_name=root_agent.name,
    description=root_agent.description,
    skills=skills,
)

# 3. Initialize A2aAgent using ADK's built-in A2aAgentExecutor
a2a_agent = A2aAgent(
    agent_card=agent_card,
    agent_executor_builder=A2aAgentExecutor,
    agent_executor_kwargs={"runner": get_reservation_runner},
    extended_agent_card=agent_card,
)
a2a_agent.set_up()
print("=== A2A Agent Successfully Initialized ===")
print(f"Name: {a2a_agent.agent_card.name}")
print(f"Skills: {[skill.name for skill in a2a_agent.agent_card.skills]}")

#### Test 1: Discovering the Agent Card

Other agents in an A2A ecosystem discover an agent's name, description, and skills by querying its agent card endpoint (`/v1/card`). We can test calling `on_get_extended_agent_card`:

In [ ]:
# 1. Test Agent Card Retrieval
card_response = await a2a_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest(),
    context=create_call_context(),
)

print("=== Agent Card ===")
print(f"Name: {card_response.name}")
print(f"Description: {card_response.description}")
print(f"Skills: {[s.name for s in card_response.skills]}")


#### Test 2: Creating a Table Reservation

In the A2A protocol, sending a message to an agent creates a **Task**. Let's send a message requesting to book a table for Saturday at 6pm. The response will return a Task object containing a unique `task_id` and an optional `contextId` that identifies the conversation session:

In [ ]:
# 2. Test Creating a Reservation via A2A Message
request = build_send_message_request(
    "Book a table for 2 on Saturday at 6pm. Name: Bob, Phone: 555-0202"
)
context = create_call_context()
response = await a2a_agent.on_message_send(request=request, context=context)

task_id = response.id
context_id = response.context_id
print(f"Task submitted! Task ID: {task_id}")
print(f"Session Context ID: {context_id}")

# Poll until the task completes
result = await wait_for_task(a2a_agent, task_id, context)
print("\n=== Reservation Creation Response ===")
print_task_answer(result)


#### Test 3: Checking the Reservation (Session Continuity)

To test multi-turn conversation and session state continuity, let's send a second message ("Check the reservation for 555-0202") passing the same `contextId`. The agent accesses ADK's `ToolContext.state` to find the reservation created in the previous turn:

In [ ]:
# 3. Test Checking the Reservation using contextId for session continuity
request = build_send_message_request(
    "Check the reservation for 555-0202",
    context_id=context_id,
)
check_response = await a2a_agent.on_message_send(
    request=request, context=context
)

check_result = await wait_for_task(a2a_agent, check_response.id, context)
print("=== Check Reservation Response ===")
print_task_answer(check_result)


#### Test 4: Cancelling the Reservation

Finally, let's test our `cancel_reservation` skill over the A2A protocol:

In [ ]:
# 4. Test Cancelling the Reservation via A2A Message
request = build_send_message_request(
    "Cancel the reservation for 555-0202",
    context_id=context_id,
)
cancel_response = await a2a_agent.on_message_send(
    request=request, context=context
)

cancel_result = await wait_for_task(a2a_agent, cancel_response.id, context)
print("=== Cancel Reservation Response ===")
print_task_answer(cancel_result)


### Deploy the Reservation Agent to Agent Runtime

In this step, you deploy your reservation agent to Google Cloud Gemini Enterprise Agent Platform Runtime — a fully managed, serverless hosting environment that exposes your agent as a secure A2A endpoint.

When deployed:
* Your code and skills are packaged and uploaded to Google Cloud Storage.
* Agent Runtime creates a managed serverless endpoint serving standard HTTP A2A endpoints (`/v1/card`, `/v1/message/send`, `/v1/tasks/get`).
* `GOOGLE_CLOUD_AGENT_ENGINE_ID` is automatically injected into the container environment, telling `A2aAgentExecutor` to switch from `InMemorySessionService` to `VertexAiSessionService` for persistent conversation sessions.

#### Deploying via `client.agent_engines.create(...)`

We use the Vertex AI Python SDK to deploy our `A2aAgent`. Because creating Google Cloud resources takes 3-5 minutes, the deployment block below is controlled by a flag `DEPLOY_TO_AGENT_RUNTIME = False`. Set it to `True` if you wish to deploy:

In [ ]:
RESOURCE_NAME = os.environ.get("RESERVATION_AGENT_RESOURCE_NAME", "")

print("Starting deployment to Agent Runtime (this takes 3-5 minutes)...")
STAGING_BUCKET = os.environ.get(
    "STAGING_BUCKET", f"{PROJECT_ID}-adk-a2a-agent-runtime"
)
BUCKET_URI = f"gs://{STAGING_BUCKET}"

vertexai.init(
    project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI
)
client = vertexai.Client(
    project=PROJECT_ID,
    location=REGION,
    http_options=types.HttpOptions(api_version="v1beta1"),
)

deploy_a2a_agent = A2aAgent(
    agent_card=agent_card,
    agent_executor_builder=A2aAgentExecutor,
    agent_executor_kwargs={"runner": get_reservation_runner},
    extended_agent_card=agent_card,
)

remote_agent = client.agent_engines.create(
    agent=deploy_a2a_agent,
    config={
        "display_name": agent_card.name,
        "description": agent_card.description,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]==1.160.0",
            "google-genai==2.11.0",
            "google-adk[mcp, a2a]==2.5.0",
            "a2a-sdk==1.1.2",
            "cloudpickle",
            "pydantic",
        ],
        "extra_packages": [
            "./a2a_agent",
        ],
        "http_options": {
            "api_version": "v1beta1",
        },
        "staging_bucket": BUCKET_URI,
    },
)

RESOURCE_NAME = remote_agent.api_resource.name
os.environ["RESERVATION_AGENT_RESOURCE_NAME"] = RESOURCE_NAME
print(f"\nDeployment complete! Resource name: {RESOURCE_NAME}")

#### Testing the Remote Deployed A2A Agent

Once deployed, any authorized client can interact with the agent over HTTP. We demonstrate fetching the remote agent card and sending a reservation request using `client.agent_engines.get(...)`:

In [ ]:
# 1. Fetch Remote Agent Card
card = await remote_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest()
)
print(f"Connected to Remote Agent: {card.name}")

# 2. Send Message to Remote Agent
print("\nSending reservation request...")
req = SendMessageRequest(
    message=new_text_message(
        "Book a table for 4 under name Alice on Friday at 7pm, phone 555-0199"
    )
)
response = await remote_agent.on_message_send(request=req)

# 3. Print Task Response
for item in response:
    task = item.task
    if hasattr(task, "artifacts") and task.artifacts:
        for art in task.artifacts:
            for part in art.parts:
                text = getattr(part, "text", None) or (
                    part.root.text if hasattr(part, "root") else None
                )
                if text:
                    print(f"\nResponse:\n{text}")


## Consuming A2A Agent


### Integrate A2A Reservation Agent with Root Restaurant Agent

In this step, we upgrade the **Foodie Finds** restaurant concierge agent (`restaurant_agent`) to consume our reservation agent as a remote A2A sub-agent using ADK's `RemoteA2aAgent`.

When integrated:
* Menu search requests ("What Italian dishes do you have?") are handled by PostgreSQL tools via MCP Toolbox.
* Reservation requests ("Book a table for 4 on Friday at 7pm") are automatically delegated to our `RemoteA2aAgent` over the A2A protocol.

#### Resolve the Agent Card URL

To delegate tasks via A2A, `RemoteA2aAgent` requires the full URL to the remote agent card (`{card.url}/v1/card`). We can resolve it dynamically from `RESOURCE_NAME` (or set a default card URL for testing):

In [ ]:
card = await remote_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest()
)
# Point the supported interface URL to our deployed Agent Runtime endpoint
card.supported_interfaces[0].url = f"https://{REGION}-aiplatform.googleapis.com/v1beta1/{RESOURCE_NAME}/a2a"
print(f"Connected to Remote Agent: {card.name}")
print(f"Resolved Endpoint URL: {card.supported_interfaces[0].url}")

RESERVATION_AGENT_CARD_URL = f"{card.supported_interfaces[0].url}"
os.environ["RESERVATION_AGENT_CARD_URL"] = RESERVATION_AGENT_CARD_URL

#### Update the Restaurant Agent (`restaurant_agent/agent.py`)

We write `./a2a_agent/restaurant_agent/agent.py` to define the `restaurant_agent`. Notice how it:
1. Defines `GoogleCloudAuth` — an auto-refreshing authentication handler for Google Cloud access tokens (`httpx.Auth`).
2. Declares `reservation_remote_agent = RemoteA2aAgent(...)` pointing to `RESERVATION_AGENT_CARD_URL`.
3. Registers `reservation_remote_agent` in `sub_agents=[reservation_remote_agent]` on the root `LlmAgent`.

When a user asks to book, check, or cancel a table, the root `restaurant_agent` delegates the request to the remote A2A reservation agent automatically!


In [ ]:
%%writefile ./a2a_agent/restaurant_agent/__init__.py
# Package init for restaurant_agent


In [ ]:
%%writefile ./a2a_agent/restaurant_agent/agent.py
# restaurant_agent/agent.py
import os

import httpx
from google.adk.agents import LlmAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.models.google_llm import Gemini
from google.auth import default
from google.auth.transport.requests import Request as AuthRequest

import re
from a2a.types import AgentCard, AgentInterface

RESERVATION_AGENT_CARD_URL = os.environ.get("RESERVATION_AGENT_CARD_URL", "")

def search_menu(cuisine_type: str = "") -> str:
    """Search restaurant menu items by cuisine type."""
    return f"Menu items for {cuisine_type or 'all cuisines'}: Margherita Pizza, Spaghetti Carbonara, Tiramisu."




def get_gcp_httpx_client(timeout: int = 60) -> httpx.AsyncClient:
    credentials, _ = default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    if not credentials.valid:
        credentials.refresh(Request())
    return httpx.AsyncClient(headers={"Authorization": f"Bearer {credentials.token}"}, timeout=timeout)


# Define remote A2A sub-agent if URL is configured
sub_agents = []
agent_card = AgentCard(
    name="reservation_agent",
    description="Handles restaurant table reservations — create, check, and cancel bookings.",
    supported_interfaces=[
        AgentInterface(
            url=RESERVATION_AGENT_CARD_URL,
            protocol_binding="HTTP+JSON",
            protocol_version="1.0",
        )
    ],
    skills=[],
)

reservation_remote_agent = RemoteA2aAgent(
    name="reservation_agent",
    description="Handles restaurant table reservations — create, check, and cancel bookings. Delegate to this agent when the user wants to book a table, check a reservation, or cancel a reservation.",
    agent_card=agent_card,
    httpx_client=get_gcp_httpx_client(),
)
sub_agents.append(reservation_remote_agent)


root_agent = LlmAgent(
    name="restaurant_agent",
    model=Gemini(
        model="gemini-3.5-flash",
        client_kwargs={"vertexai": True, "location": "global"},
    ),
    instruction="""You are a friendly and knowledgeable concierge at "Foodie Finds," a restaurant. Your job:
- Help diners browse the menu by category or cuisine type.
- Provide full details about specific dishes, including ingredients, price, and dietary information.
- Recommend dishes based on natural language descriptions of what the diner is craving.
- Add new menu items when asked.
- For reservation requests (booking, checking, or cancelling tables), delegate to the reservation_agent.

When a diner asks about a specific dish by name or cuisine, use the menu tools.
For any reservation requests, always delegate to the reservation_agent.
Be conversational, knowledgeable, and concise.""",
    tools=[search_menu],
    sub_agents=sub_agents,
)


In [ ]:
%%writefile ./a2a_agent/restaurant_agent/.env
# Specify global location for Gemini models and Vertex AI ADC
GOOGLE_CLOUD_LOCATION=global
GOOGLE_GENAI_USE_VERTEXAI=TRUE


#### Test the Integrated Agent
You can test the integrated concierge agent using the ADK Developer UI (`adk web`):

In the developer UI, try:
* **Menu query**: *"What Italian dishes do you have?"* — handled by the menu search tools.
* **Reservation request**: *"I want to create a reservation under name Bob, phone number 123456"* — notice how the orchestrator delegates to `reservation_agent` via the A2A protocol!
* **Check reservation**: *"Check the reservation for 123456"* — delegated to `reservation_agent`.

In [ ]:
# On Cloud Workstations
!adk web a2a_agent/restaurant_agent --allow_origins "regex:https://.*\.cloudworkstations\.dev"

**Note**: If you are using Agent Platform Workbench, remove the comment out and run the cell below.



In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000"
# echo "--------------------------------------------------------"
# adk web custom_mcp_agent --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"


Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.